# 02. DataFrame, Series, типы данных и базовая проверка качества

## Тема

**Загрузка и интеграция данных из различных форматов. Инструменты для сбора данных. Основы Python для обработки данных**

В предыдущем ноутбуке мы научились загружать данные из CSV, Excel, JSON и HTML.  
Теперь разберемся, что делать **после загрузки**:

- что такое `DataFrame`;
- что такое `Series`;
- как смотреть первые и последние строки;
- как проверять размер таблицы и столбцы;
- как понимать `info()` и `dtypes`;
- как преобразовывать типы данных;
- как работать со строками;
- как находить пропуски;
- как находить дубликаты;
- как выполнять базовую проверку качества данных.

Часть идей сохранена из исходного файла `part_1.ipynb`: создание `DataFrame`, работа с `Series`, проверка типов, `.str.lower()` и `.astype()`.

## 1. Цель ноутбука

После выполнения этого ноутбука вы должны уметь:

1. Объяснять, что такое `DataFrame`.
2. Объяснять, что такое `Series`.
3. Использовать `head()`, `tail()`, `shape`, `columns`.
4. Использовать `info()` и `dtypes`.
5. Приводить типы данных через `astype()`.
6. Преобразовывать даты через `pd.to_datetime()`.
7. Преобразовывать числа через `pd.to_numeric()`.
8. Очищать строки через `.str.strip()`, `.str.lower()`, `.str.upper()`.
9. Находить пропуски через `isna()`.
10. Находить дубликаты через `duplicated()`.
11. Делать базовую проверку качества данных перед анализом.

## 2. Импорт библиотек и поиск данных

Для работы нам нужны:

- `pandas` — основная библиотека для таблиц;
- `Path` — удобная работа с путями к файлам.

In [ ]:
import pandas as pd
from pathlib import Path

print("pandas:", pd.__version__)

In [ ]:
def find_data_dir() -> Path:
    """Найти папку с учебными данными."""
    current_dir = Path.cwd()

    candidates = [
        current_dir / "data" / "raw",
        current_dir.parent / "data" / "raw",
        current_dir.parent.parent / "data" / "raw",
    ]

    for candidate in candidates:
        if (candidate / "sales.csv").exists():
            return candidate

    return current_dir / "data" / "raw"


DATA_DIR = find_data_dir()

print("Папка с данными:")
print(DATA_DIR)

## 3. Загрузка исходных таблиц

В этом ноутбуке будем работать в основном с `sales.csv`, но дополнительно загрузим `products.xlsx` и `clients.csv`, чтобы показать проверки справочников.

In [ ]:
sales = pd.read_csv(DATA_DIR / "sales.csv")
products = pd.read_excel(DATA_DIR / "products.xlsx", sheet_name="products")
clients = pd.read_csv(DATA_DIR / "clients.csv")

print("sales:", sales.shape)
print("products:", products.shape)
print("clients:", clients.shape)

# Часть 1. DataFrame и Series

## 4. Что такое DataFrame

`DataFrame` — это таблица в pandas.

У таблицы есть:

- строки;
- столбцы;
- значения в ячейках;
- типы данных у столбцов.

Сначала создадим маленький пример, как в исходном `part_1.ipynb`.

In [ ]:
demo_df = pd.DataFrame({
    "Числа": [1, 2, 3],
    "Строки": ["Один", "Два", "Три"],
    "Логические": [True, False, True],
})

demo_df

### Что важно заметить

В одном `DataFrame` могут быть столбцы разных типов:

- числа;
- строки;
- логические значения.

In [ ]:
print("Тип объекта demo_df:")
print(type(demo_df))

print("\nТипы данных столбцов:")
print(demo_df.dtypes)

## 5. Что такое Series

`Series` — это один столбец pandas.

Если `DataFrame` — это таблица, то `Series` — это один столбец этой таблицы.

Возьмем столбец `Числа` из маленького примера.

In [ ]:
numbers_series = demo_df["Числа"]

print(numbers_series)
print("\nТип объекта:")
print(type(numbers_series))

print("\nТип данных внутри Series:")
print(numbers_series.dtype)

## 6. Series из реального датасета

Теперь возьмем столбец `channel` из таблицы продаж.

Это уже не искусственный пример, а реальные данные нашего учебного кейса.

In [ ]:
channel_series = sales["channel"]

print(channel_series.head())
print("\nТип объекта:", type(channel_series))
print("Тип данных:", channel_series.dtype)

### Пояснение

`channel` — это канал продаж:

- online;
- offline;
- marketplace.

Но в учебных данных специально есть проблемы: лишние пробелы и разный регистр.

# Часть 2. Первичный осмотр DataFrame

## 7. `head()` — первые строки

`head()` показывает первые строки таблицы.

Обычно это первая команда после загрузки данных.

In [ ]:
sales.head()

## 8. `tail()` — последние строки

`tail()` показывает последние строки таблицы.

Это помогает проверить конец файла: иногда ошибки встречаются именно там.

In [ ]:
sales.tail()

## 9. `shape` — размер таблицы

`shape` показывает количество строк и столбцов.

Формат результата:

```text
(строки, столбцы)
```

In [ ]:
sales.shape

In [ ]:
rows, columns = sales.shape

print("Количество строк:", rows)
print("Количество столбцов:", columns)

## 10. `columns` — список столбцов

`columns` показывает названия столбцов.

Это важно перед фильтрацией, расчетами и объединением таблиц.

In [ ]:
sales.columns

In [ ]:
sales.columns.tolist()

## 11. `info()` — общая информация о таблице

`info()` показывает:

- названия столбцов;
- количество непустых значений;
- типы данных;
- примерный объем памяти.

Это одна из самых полезных команд после загрузки файла.

In [ ]:
sales.info()

## 12. `dtypes` — типы данных столбцов

`dtypes` показывает тип каждого столбца.

Тип данных определяет, какие операции можно делать со столбцом.

In [ ]:
sales.dtypes

### Основные типы, которые часто встречаются

| Тип | Что означает |
|---|---|
| `int64` | целые числа |
| `float64` | числа с дробной частью |
| `object` | чаще всего текст или смешанные значения |
| `bool` | логические значения `True` / `False` |
| `datetime64` | дата и время |
| `category` | категориальные значения |

Важно: если числовой столбец загрузился как `object`, возможно, в нем есть текстовые ошибки.

# Часть 3. Преобразование типов данных

## 13. Простой пример `astype()`

Сохраняем идею из исходного `part_1.ipynb`: сначала преобразуем возраст в строку, потом обратно в число.

In [ ]:
age_df = pd.DataFrame({"Age": [25, 30, 35]})

print("Исходная таблица:")
display(age_df)

print("Тип данных Age:")
print(age_df["Age"].dtype)

In [ ]:
age_df["Age"] = age_df["Age"].astype(str)

print("После преобразования в строку:")
display(age_df)

print("Тип данных Age:")
print(age_df["Age"].dtype)

In [ ]:
age_df["Age"] = age_df["Age"].astype(int)

print("После обратного преобразования в число:")
display(age_df)

print("Тип данных Age:")
print(age_df["Age"].dtype)

## 14. Почему `astype()` не всегда срабатывает

Если в столбце есть текстовая ошибка, прямое преобразование в число может упасть.

В `sales.csv` специально есть значение `price_error` в столбце `unit_price`.

In [ ]:
sales["unit_price"].head(10)

In [ ]:
# Эта ячейка специально показывает возможную ошибку.
# Если выполнить прямое преобразование, pandas не сможет превратить 'price_error' в число.

try:
    sales["unit_price"].astype(float)
except ValueError as error:
    print("Ошибка преобразования:")
    print(error)

### Что делать вместо прямого `astype(float)`

Для грязных числовых данных часто используют:

```python
pd.to_numeric(..., errors="coerce")
```

`errors="coerce"` означает:

> если значение нельзя превратить в число, замени его на пропуск `NaN`.

In [ ]:
sales["unit_price_num"] = pd.to_numeric(sales["unit_price"], errors="coerce")

sales[["unit_price", "unit_price_num"]].head(10)

In [ ]:
print("Количество значений unit_price, которые не удалось преобразовать в число:")
print(sales["unit_price_num"].isna().sum())

## 15. Преобразование скидки в число

В `discount_percent` могут быть:

- обычные числа;
- пропуски;
- текстовое значение `five`;
- слишком большая скидка `150`.

Сначала преобразуем в число.

In [ ]:
sales["discount_percent_num"] = pd.to_numeric(sales["discount_percent"], errors="coerce")

sales[["discount_percent", "discount_percent_num"]].head(15)

In [ ]:
print("Пропуски после преобразования discount_percent:")
print(sales["discount_percent_num"].isna().sum())

## 16. Преобразование даты через `pd.to_datetime()`

Дата из CSV часто загружается как текст.

В нашем датасете специально есть разные форматы дат:

- `2026-01-12`;
- `12.01.2026`;
- `2026/01/13`;
- `14-01-2026`;
- возможно, ошибочные значения.

Преобразуем `order_date` к типу даты.

In [ ]:
sales["order_date"].head(10)

In [ ]:
def parse_dates_safely(series: pd.Series) -> pd.Series:
    """Преобразовать даты с учетом разных версий pandas."""
    try:
        return pd.to_datetime(series, errors="coerce", format="mixed", dayfirst=True)
    except TypeError:
        return pd.to_datetime(series, errors="coerce", dayfirst=True)


sales["order_date_dt"] = parse_dates_safely(sales["order_date"])

sales[["order_date", "order_date_dt"]].head(15)

In [ ]:
print("Тип данных order_date_dt:")
print(sales["order_date_dt"].dtype)

print("\nКоличество дат, которые не удалось распознать:")
print(sales["order_date_dt"].isna().sum())

## 17. Создание месяца из даты

После преобразования даты можно извлекать из нее части:

- год;
- месяц;
- день;
- день недели.

Создадим поле `month`.

In [ ]:
sales["month"] = sales["order_date_dt"].dt.to_period("M").astype(str)

sales[["order_date", "order_date_dt", "month"]].head()

# Часть 4. Работа со строками

## 18. Строковые операции `.str`

В исходном `part_1.ipynb` была идея с преобразованием имен к нижнему регистру. Сохраним ее и расширим.

In [ ]:
names_df = pd.DataFrame({"Name": ["alice", "BOB", "Charlie"]})

print("Исходная таблица:")
display(names_df)

names_df["Name"] = names_df["Name"].str.lower()

print("После str.lower():")
display(names_df)

## 19. Очистка строк в реальном датасете

Посмотрим уникальные значения `channel` до очистки.

In [ ]:
sales["channel"].unique()

Можно заметить проблемы:

- разный регистр: `online`, `Online`, `ONLINE`;
- лишние пробелы: например, `online `.

Очистим значения.

In [ ]:
sales["channel_clean"] = (
    sales["channel"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print("До очистки:")
print(sales["channel"].unique())

print("\nПосле очистки:")
print(sales["channel_clean"].unique())

## 20. Очистка `client_type` в справочнике клиентов

В `clients.csv` также есть разные варианты написания типа клиента:

- `B2C`;
- `b2c`;
- `B2B `.

Приведем к единому виду.

In [ ]:
print("До очистки:")
print(clients["client_type"].unique())

clients["client_type_clean"] = (
    clients["client_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print("\nПосле очистки:")
print(clients["client_type_clean"].unique())

## 21. Очистка категорий товаров

Посмотрим категории в `products.xlsx`.

Там специально есть лишние пробелы и разный регистр.

In [ ]:
print("До очистки:")
print(products["category"].unique())

products["category_clean"] = (
    products["category"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print("\nПосле очистки:")
print(products["category_clean"].unique())

# Часть 5. Пропуски

## 22. Что такое пропуск

Пропуск — это отсутствие значения.

В pandas пропуски часто отображаются как:

- `NaN`;
- `NaT` для дат;
- пустые значения после загрузки файла.

Найдем пропуски в таблице продаж.

In [ ]:
sales.isna().sum()

## 23. Пропуски в конкретных столбцах

Посмотрим строки, где не удалось преобразовать `unit_price` в число.

In [ ]:
sales[sales["unit_price_num"].isna()][["sale_id", "unit_price", "unit_price_num"]]

Посмотрим строки, где скидка стала пропуском после преобразования.

In [ ]:
sales[sales["discount_percent_num"].isna()][["sale_id", "discount_percent", "discount_percent_num"]].head(10)

## 24. Заполнение пропусков

Для скидки можно принять бизнес-правило:

> если скидка не указана, считаем ее равной 0.

Важно: это не универсальное правило. Это решение должно быть согласовано с бизнес-логикой.

In [ ]:
sales["discount_percent_fixed"] = sales["discount_percent_num"].fillna(0)

sales[["discount_percent", "discount_percent_num", "discount_percent_fixed"]].head(15)

# Часть 6. Дубликаты

## 25. Поиск полных дубликатов строк

Полный дубликат — это строка, которая полностью совпадает с другой строкой.

In [ ]:
print("Количество полных дубликатов строк:")
print(sales.duplicated().sum())

In [ ]:
sales[sales.duplicated(keep=False)].sort_values("sale_id")

## 26. Дубликаты по ключевому полю

Иногда строки не полностью одинаковые, но повторяется идентификатор.

Для таблицы продаж важен `sale_id`.

In [ ]:
print("Количество дубликатов по sale_id:")
print(sales.duplicated(subset=["sale_id"]).sum())

In [ ]:
sales[sales.duplicated(subset=["sale_id"], keep=False)].sort_values("sale_id")

## 27. Удаление дубликатов

В учебных целях создадим новую таблицу без дубликатов по `sale_id`.

Важно: не будем перезаписывать исходную таблицу `sales`, чтобы можно было сравнивать результат.

In [ ]:
sales_no_duplicates = sales.drop_duplicates(subset=["sale_id"], keep="first")

print("Было строк:", sales.shape[0])
print("Стало строк:", sales_no_duplicates.shape[0])

# Часть 7. Базовая проверка качества данных

## 28. Проверка диапазонов

Проверим бизнес-аномалии:

- количество товара меньше или равно 0;
- скидка меньше 0 или больше 100;
- цена не преобразовалась в число.

In [ ]:
bad_quantity = sales[pd.to_numeric(sales["quantity"], errors="coerce") <= 0]
bad_discount = sales[
    (sales["discount_percent_fixed"] < 0) | 
    (sales["discount_percent_fixed"] > 100)
]
bad_price = sales[sales["unit_price_num"].isna()]

print("Строк с quantity <= 0:", bad_quantity.shape[0])
print("Строк со скидкой вне диапазона 0–100:", bad_discount.shape[0])
print("Строк с ошибочной ценой:", bad_price.shape[0])

## 29. Просмотр проблемных строк

Посмотрим проблемные строки отдельно.

In [ ]:
bad_quantity[["sale_id", "quantity", "order_status"]]

In [ ]:
bad_discount[["sale_id", "discount_percent", "discount_percent_fixed"]]

In [ ]:
bad_price[["sale_id", "unit_price", "unit_price_num"]]

## 30. Проверка справочника товаров

В справочниках тоже бывают ошибки.

Проверим:

- дубликаты `product_id`;
- пропуски в `supplier`;
- ошибки в `purchase_price`.

In [ ]:
products["purchase_price_num"] = pd.to_numeric(products["purchase_price"], errors="coerce")

print("Дубликаты product_id:")
print(products.duplicated(subset=["product_id"]).sum())

print("\nПропуски supplier:")
print(products["supplier"].isna().sum())

print("\nОшибки purchase_price:")
print(products["purchase_price_num"].isna().sum())

In [ ]:
products[
    products.duplicated(subset=["product_id"], keep=False) |
    products["supplier"].isna() |
    products["purchase_price_num"].isna()
]

## 31. Проверка ключей перед объединением

Даже до `merge` можно проверить, все ли `product_id` из продаж есть в справочнике товаров.

Это базовая проверка качества ключей.

In [ ]:
sales_product_ids = set(sales["product_id"].dropna().unique())
product_reference_ids = set(products["product_id"].dropna().unique())

missing_product_ids = sales_product_ids - product_reference_ids

print("product_id из продаж, которых нет в справочнике товаров:")
print(missing_product_ids)

# Часть 8. Мини-пайплайн первичной очистки

## 32. Создадим рабочую копию таблицы

В реальной работе лучше не портить исходные данные. Поэтому создаем копию.

In [ ]:
sales_clean = sales.copy()

sales_clean.head()

## 33. Применим базовую очистку

Сделаем:

- дату в datetime;
- цену в число;
- скидку в число;
- пропущенную скидку заменим на 0;
- канал продаж очистим от пробелов и регистра;
- удалим дубликаты по `sale_id`.

In [ ]:
sales_clean["order_date"] = parse_dates_safely(sales_clean["order_date"])
sales_clean["unit_price"] = pd.to_numeric(sales_clean["unit_price"], errors="coerce")
sales_clean["discount_percent"] = pd.to_numeric(sales_clean["discount_percent"], errors="coerce").fillna(0)
sales_clean["channel"] = sales_clean["channel"].astype("string").str.strip().str.lower()

sales_clean = sales_clean.drop_duplicates(subset=["sale_id"], keep="first")

sales_clean.head()

## 34. Проверим результат очистки

Сравним типы и основные проблемы после очистки.

In [ ]:
sales_clean.info()

In [ ]:
print("Пропуски после базовой очистки:")
print(sales_clean.isna().sum())

print("\nУникальные channel после очистки:")
print(sales_clean["channel"].unique())

print("\nДубликаты sale_id после очистки:")
print(sales_clean.duplicated(subset=["sale_id"]).sum())

## 35. Создадим расчетное поле `gross_revenue`

Теперь, когда цена и количество приведены к числовому виду, можно посчитать выручку до скидки.

In [ ]:
sales_clean["quantity"] = pd.to_numeric(sales_clean["quantity"], errors="coerce")
sales_clean["gross_revenue"] = sales_clean["quantity"] * sales_clean["unit_price"]

sales_clean[["sale_id", "quantity", "unit_price", "gross_revenue"]].head(10)

# Часть 9. Мини-задания

Выполните задания самостоятельно.

## Задание 1

Выведите первые 7 строк таблицы `sales`.

In [ ]:
# Ваш код здесь

## Задание 2

Выведите последние 3 строки таблицы `products`.

In [ ]:
# Ваш код здесь

## Задание 3

Выведите количество строк и столбцов в таблице `clients`.

In [ ]:
# Ваш код здесь

## Задание 4

Выведите список столбцов таблицы `sales`.

In [ ]:
# Ваш код здесь

## Задание 5

Проверьте типы данных в таблице `sales`.

In [ ]:
# Ваш код здесь

## Задание 6

Создайте столбец `channel_task`, где `channel` очищен от пробелов и приведен к нижнему регистру.

In [ ]:
# Ваш код здесь

## Задание 7

Преобразуйте `discount_percent` в число в новом столбце `discount_task`.

Ошибки преобразования замените на `NaN`.

In [ ]:
# Ваш код здесь

## Задание 8

Найдите дубликаты по `sale_id`.

In [ ]:
# Ваш код здесь

## Задание 9

Найдите строки, где `quantity` меньше или равно 0.

In [ ]:
# Ваш код здесь

## Задание 10

Проверьте, какие `client_id` из продаж отсутствуют в справочнике `clients`.

In [ ]:
# Ваш код здесь

# Часть 10. Контрольные вопросы

Ответьте своими словами:

1. Чем `DataFrame` отличается от `Series`?
2. Что показывает `head()`?
3. Что показывает `tail()`?
4. Что показывает `shape`?
5. Зачем проверять `columns`?
6. Чем `info()` отличается от `dtypes`?
7. Для чего нужен `astype()`?
8. Почему иногда лучше использовать `pd.to_numeric(..., errors="coerce")`, а не `astype(float)`?
9. Что делает `pd.to_datetime()`?
10. Что делают `.str.strip()` и `.str.lower()`?
11. Как найти пропуски?
12. Как найти дубликаты?
13. Почему перед расчетами нужно проверять качество данных?

# 36. Итог ноутбука

В этом ноутбуке мы научились:

- понимать структуру `DataFrame`;
- работать с отдельным столбцом как с `Series`;
- смотреть первые и последние строки;
- проверять размер таблицы и список столбцов;
- анализировать типы данных;
- преобразовывать строки, числа и даты;
- находить пропуски;
- находить дубликаты;
- выполнять базовую проверку качества данных.

Следующий шаг:

```text
03_data_integration.ipynb
```

Там мы будем объединять несколько таблиц: продажи, товары, регионы и клиентов.